In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

In [17]:
data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\Cancer Prediction v2. - DS1 (1).csv" 
df = pd.read_csv(data_path, delimiter=",")  

# Features and target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]


print(np.bincount(y))
smote = SMOTE(random_state=42, k_neighbors=5)
X, y = smote.fit_resample(X, y)
print(np.bincount(y))


# (7 : 1 : 2)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_cv, y_train, y_cv = train_test_split(X_temp, y_temp, test_size=1/8, random_state=42, stratify=y_temp)

#Before SMOTE
print(np.bincount(y_train), np.bincount(y_cv), np.bincount(y_test))
print(y_train.shape, y_cv.shape, y_test.shape)

[234  75]
[234 234]
[143 143] [20 21] [71 70]
(286,) (41,) (141,)


In [16]:
external_test_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project .v2\ET1 (1).csv"
df_external = pd.read_csv(external_test_path, delimiter=",")
X_external = df_external.iloc[:, :-1]
y_external = df_external.iloc[:, -1]

print(X_external)
print(y_external)

     200717_x_at  202192_s_at  203592_s_at  207574_s_at  209304_x_at  \
0       0.780951    -0.336587    -0.255506     0.550705     0.356276   
1      -0.351034    -1.208983    -0.688482    -2.158107    -2.155740   
2      -0.432727    -0.343613    -0.115302    -0.542080     0.240398   
3      -1.260738    -0.403174     0.496536    -1.772841    -1.350446   
4       0.809847     2.112079     0.310269     0.242549     0.279259   
..           ...          ...          ...          ...          ...   
240     1.578388     0.149185    -0.287055     2.202842     2.375767   
241    -0.107202     1.306598     0.238374     0.806519     0.838113   
242     1.846280    -0.008404    -0.524969    -0.592765    -0.387908   
243     0.791526     0.319873    -0.239817    -0.231235    -0.034546   
244    -0.060069     0.173836    -0.064435     0.363354     0.123174   

     212356_at  218686_s_at  219173_at  37796_at  
0    -1.011282     0.780860  -0.192744 -1.359188  
1    -0.252015    -0.978074   0.5

In [20]:
# === Filter + Wrapper Feature Selection with Multiple Models ===
from sklearn.feature_selection import SelectKBest, f_classif, RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

# Define base models
models = {
    # "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', random_state=42)),
    # "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    # "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
    #                           use_label_encoder=False, eval_metric='logloss', random_state=42),
    # "Bagging_RF": BaggingClassifier(
    #     estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    #     n_estimators=10, random_state=42
    # ),
    # "SVM": make_pipeline(StandardScaler(), SVC(probability=True, kernel='rbf', C=1.0, random_state=42)),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, random_state=42)
}

# Filter method config
filter_k = 200  # Tune based on total gene count

# Loop through each model
for name, model in models.items():
    print(f"\n==== Model: {name} ====")

    # Step 1: Filter-based selection
    filter_selector = SelectKBest(score_func=f_classif, k=filter_k)
    wrapper_selector = RFECV(
        estimator=LogisticRegression(solver='liblinear'),
        step=1,
        cv=StratifiedKFold(5),
        scoring='roc_auc',
        n_jobs=-1
    )

    # Combined feature selection pipeline
    feature_pipeline = Pipeline([
        ('filter', filter_selector),
        ('wrapper', wrapper_selector)
    ])

    # Apply feature selection
    X_train_selected = feature_pipeline.fit_transform(X_train, y_train)
    X_val_selected = feature_pipeline.transform(X_cv)
    X_test_selected = feature_pipeline.transform(X_test)
    X_external_selected = feature_pipeline.transform(X_external)

    # Fit the model
    model.fit(X_train_selected, y_train)

    selected_feature_names_filter = X.columns[filter_selector.get_support()]
    selected_feature_names_wrapper = selected_feature_names_filter[wrapper_selector.get_support()]

    print("Selected features after RFECV:")
    print(selected_feature_names_wrapper)

    # Evaluate on all sets
    for split_name, X_set, y_set in zip(
        ['Train', 'Validation', 'Internal Test', 'External Test'],
        [X_train_selected, X_val_selected, X_test_selected, X_external_selected],
        [y_train, y_cv, y_test, y_external]
    ):
        probas = model.predict_proba(X_set)[:, 1]
        preds = model.predict(X_set)

        auc = roc_auc_score(y_set, probas)
        acc = accuracy_score(y_set, preds)

        print(f"{split_name} - AUC: {auc:.4f} | Accuracy: {acc:.4f}")



==== Model: ExtraTrees ====


c:\Users\Win\Documents\Github Workspace\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:783: UserWarning: k=200 is greater than n_features=9. All the features will be returned.
  warnings.warn(


Selected features after RFECV:
Index(['200717_x_at', '202192_s_at', '203592_s_at', '207574_s_at',
       '209304_x_at', '212356_at', '218686_s_at', '219173_at', '37796_at'],
      dtype='object')
Train - AUC: 1.0000 | Accuracy: 1.0000
Validation - AUC: 0.9655 | Accuracy: 0.8293
Internal Test - AUC: 0.9483 | Accuracy: 0.8440
External Test - AUC: 0.9973 | Accuracy: 0.9265
